# Day 1 - Neural Network Foundations & Your First MLP

**Module:** Deep Learning with AI APIs (Day 1 of 4)

Companion notebook for `03_lecture/day1_slides.md`. We move from a single neuron in NumPy all the way to a Keras MLP that hits ~98% on MNIST.

**Sections:**
1. Setup & imports
2. Part A - A neuron from scratch (NumPy)
3. Part B - Why we need hidden layers (XOR)
4. Part C - Training: gradient descent by hand
5. Part D - Your first Keras MLP on MNIST
6. Part E - Diagnosing overfitting & adding Dropout
7. Exercises

## 1. Setup & imports

We use NumPy for the from-scratch parts and TensorFlow/Keras for the MNIST model. If TensorFlow is not installed, run:

```bash
pip install tensorflow matplotlib
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

np.random.seed(0)
tf.random.set_seed(0)

print('NumPy     :', np.__version__)
print('TensorFlow:', tf.__version__)
print('Keras     :', keras.__version__)

## 2. Part A - A neuron from scratch (NumPy)

A neuron does three things: multiply inputs by weights, add a bias, and pass the result through an activation. That's it.

$$z = w_1 x_1 + w_2 x_2 + \dots + b \qquad a = f(z)$$

Below: a 5-line neuron that classifies points on a 2D plane.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def neuron(X, w, b):
    """One neuron: weighted sum + bias + sigmoid."""
    return sigmoid(X @ w + b)

# A toy linearly-separable dataset: points above the line y = x are class 1.
X = np.array([[0.1, 0.9], [0.2, 0.8], [0.9, 0.1], [0.8, 0.2], [0.3, 0.7], [0.7, 0.3]])
y = np.array([1, 1, 0, 0, 1, 0])

# Hand-picked weights that implement 'is x2 > x1?'
w = np.array([-1.0, 1.0])
b = 0.0

preds = (neuron(X, w, b) > 0.5).astype(int)
print('Predictions:', preds)
print('Truth      :', y)
print('Accuracy   :', (preds == y).mean())

One neuron draws **one straight line** through input space. That works here because the data is linearly separable. Now let's break it.

## 3. Part B - Why we need hidden layers (XOR)

XOR returns 1 only if the inputs differ. There is **no straight line** that separates the 1s from the 0s.

| x1 | x2 | y |
|----|----|---|
| 0  | 0  | 0 |
| 0  | 1  | 1 |
| 1  | 0  | 1 |
| 1  | 1  | 0 |

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)

# Try every weight combination of a single neuron - it will never get all 4 right.
best_acc = 0.0
for _ in range(2000):
    w = np.random.randn(2)
    b = np.random.randn()
    preds = (neuron(X_xor, w, b) > 0.5).astype(int)
    acc = (preds == y_xor).mean()
    best_acc = max(best_acc, acc)

print(f'Best accuracy a single neuron can achieve on XOR (random search): {best_acc:.2f}')
print('It tops out at 0.75 - one of the four points is always wrong.')

Now let's add a **hidden layer** of two neurons. With hand-picked weights we can solve XOR exactly. The hidden layer transforms the input into a new space where the problem becomes linearly separable.

In [ ]:
def relu(z):
    return np.maximum(0.0, z)

# Hidden layer: 2 neurons. Output layer: 1 neuron.
W1 = np.array([[1.0, 1.0],
               [1.0, 1.0]])
b1 = np.array([0.0, -1.0])
W2 = np.array([[1.0], [-2.0]])
b2 = np.array([0.0])

H = relu(X_xor @ W1 + b1)
Y_hat = sigmoid(H @ W2 + b2).ravel()
preds = (Y_hat > 0.5).astype(int)

print('Inputs :', X_xor.tolist())
print('Hidden :')
print(H)
print('Y_hat  :', np.round(Y_hat, 3))
print('Preds  :', preds)
print('Truth  :', y_xor.astype(int))
print('Accuracy:', (preds == y_xor).mean())

Hidden layers create **new features**. That is the whole point of going deep.

## 4. Part C - Training: gradient descent by hand

Above we picked the weights ourselves. Real networks **learn** them by:

1. Forward pass -> predictions
2. Compute the loss (binary cross-entropy here)
3. Backward pass -> gradients of loss w.r.t. each weight
4. Update: `w := w - learning_rate * gradient`

Below we run this loop on XOR. We give you the gradient formulas - you do not need to re-derive them. (In Keras you never write these by hand again.)

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def relu(z):
    return np.maximum(0.0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

rng = np.random.default_rng(42)
X = X_xor
y = y_xor.reshape(-1, 1)

# 2 -> 4 -> 1 network
W1 = rng.normal(size=(2, 4)) * 0.5
b1 = np.zeros((1, 4))
W2 = rng.normal(size=(4, 1)) * 0.5
b2 = np.zeros((1, 1))

lr = 0.1
losses = []

for step in range(5000):
    # ---- forward ----
    Z1 = X @ W1 + b1
    A1 = relu(Z1)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)

    # ---- loss (binary cross-entropy) ----
    eps = 1e-9
    loss = -np.mean(y * np.log(A2 + eps) + (1 - y) * np.log(1 - A2 + eps))
    losses.append(loss)

    # ---- backward (provided) ----
    dZ2 = (A2 - y) / X.shape[0]
    dW2 = A1.T @ dZ2
    db2 = dZ2.sum(axis=0, keepdims=True)

    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * relu_deriv(Z1)
    dW1 = X.T @ dZ1
    db1 = dZ1.sum(axis=0, keepdims=True)

    # ---- update ----
    W2 -= lr * dW2; b2 -= lr * db2
    W1 -= lr * dW1; b1 -= lr * db1

    if step % 1000 == 0:
        print(f'step {step:4d}  loss={loss:.4f}')

print('\nFinal predictions:')
print(np.round(A2.ravel(), 3))
print('Truth:', y.ravel().astype(int))

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.title('XOR training loss (hand-rolled gradient descent)')
plt.xlabel('step'); plt.ylabel('binary cross-entropy')
plt.grid(True)
plt.show()

The loss drops from ~0.7 to nearly 0 - the network learned XOR from scratch with only NumPy.

**Important:** in Keras you never write the backward pass yourself. The framework does it automatically with `tf.GradientTape` under the hood. The next sections leave the math behind and use Keras.

## 5. Part D - Your first Keras MLP on MNIST

MNIST: 70 000 grayscale 28x28 images of digits 0-9. The "hello world" of deep learning.

Pipeline:

1. Load and normalize.
2. Flatten images to 784-dim vectors.
3. Define a 2-hidden-layer MLP.
4. Compile with Adam + sparse categorical cross-entropy.
5. Fit with `validation_split=0.1`.
6. Plot training curves and evaluate on test set.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print('x_train shape:', x_train.shape, 'y_train shape:', y_train.shape)
print('x_test  shape:', x_test.shape,  'y_test  shape:', y_test.shape)

# Show a few samples
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, img, label in zip(axes, x_train[:6], y_train[:6]):
    ax.imshow(img, cmap='gray')
    ax.set_title(int(label)); ax.axis('off')
plt.show()

In [ ]:
# Flatten + normalize to [0, 1]
x_train_f = x_train.reshape(-1, 784).astype('float32') / 255.0
x_test_f  = x_test.reshape(-1, 784).astype('float32')  / 255.0

print('x_train_f:', x_train_f.shape, 'min', x_train_f.min(), 'max', x_train_f.max())

### Define -> compile -> fit

The Keras 3-step recipe.

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(784,)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(64,  activation='relu'),
    keras.layers.Dense(10,  activation='softmax'),
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

In [ ]:
history = model.fit(
    x_train_f, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=2,
)

In [ ]:
test_loss, test_acc = model.evaluate(x_test_f, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}')

### Reading the training curves

- Train loss should fall steadily.
- Val loss should fall, then flatten.
- If val loss starts climbing while train loss keeps falling -> **overfitting**.

In [ ]:
def plot_history(hist, title='training curves'):
    h = hist.history
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].plot(h['loss'], label='train')
    axes[0].plot(h['val_loss'], label='val')
    axes[0].set_title(f'{title} - loss'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(h['accuracy'], label='train')
    axes[1].plot(h['val_accuracy'], label='val')
    axes[1].set_title(f'{title} - accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(True)
    plt.tight_layout(); plt.show()

plot_history(history, 'baseline MLP')

## 6. Part E - Diagnosing overfitting & adding Dropout

If the gap between train and val accuracy starts to grow, the network is memorising. **Dropout** randomly zeros out a fraction of activations during training, which discourages over-reliance on any single neuron.

In [ ]:
model_dp = keras.Sequential([
    keras.layers.Input(shape=(784,)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(64,  activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(10,  activation='softmax'),
])

model_dp.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

history_dp = model_dp.fit(
    x_train_f, y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    verbose=2,
)

test_loss_dp, test_acc_dp = model_dp.evaluate(x_test_f, y_test, verbose=0)
print(f'Test accuracy (with Dropout): {test_acc_dp:.4f}')

In [ ]:
plot_history(history_dp, 'MLP + Dropout')

### Look at the mistakes

The single best debugging habit: look at examples your model gets wrong.

In [ ]:
probs = model_dp.predict(x_test_f, verbose=0)
pred = probs.argmax(axis=1)
wrong_idx = np.where(pred != y_test)[0]
print(f'Total wrong: {len(wrong_idx)} out of {len(y_test)}')

show = wrong_idx[:5]
fig, axes = plt.subplots(1, 5, figsize=(11, 2.5))
for ax, i in zip(axes, show):
    ax.imshow(x_test[i], cmap='gray')
    ax.set_title(f'pred {pred[i]}\ntrue {y_test[i]}')
    ax.axis('off')
plt.suptitle('Misclassified test digits')
plt.show()

Many of these are genuinely ambiguous - even a human would hesitate. That tells you the model is doing well, and that the remaining errors are mostly hard cases.

## 7. Exercises

Edit the code cells below. Try one at a time, observe what changes.

### TODO 1 - Change the hidden layer size

Re-define the model with the **first hidden layer set to 256 units** (instead of 128). Keep everything else the same. Train and report:

- Test accuracy
- Number of parameters (`model.summary()`)
- Subjective: did training take noticeably longer?

In [ ]:
# TODO 1: build a model with 256 units in the first hidden layer
# model_256 = keras.Sequential([
#     keras.layers.Input(shape=(784,)),
#     keras.layers.Dense(...),  # <-- 256 here
#     keras.layers.Dense(64, activation='relu'),
#     keras.layers.Dense(10, activation='softmax'),
# ])
# model_256.compile(...)
# model_256.summary()
# history_256 = model_256.fit(x_train_f, y_train, epochs=10, batch_size=128, validation_split=0.1, verbose=2)
# print('Test accuracy:', model_256.evaluate(x_test_f, y_test, verbose=0)[1])

### TODO 2 - Swap the optimizer

Re-train the original architecture using **`sgd`** instead of `adam`. Compare loss curves with the Adam run. Which converges faster?

In [ ]:
# TODO 2: same architecture, optimizer='sgd'
# model_sgd = keras.Sequential([...])
# model_sgd.compile(optimizer='sgd', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# history_sgd = model_sgd.fit(x_train_f, y_train, epochs=10, batch_size=128, validation_split=0.1, verbose=2)
# plot_history(history_sgd, 'SGD')

### TODO 3 - Swap to Fashion-MNIST

Same image shape (28x28, 10 classes), but the items are clothes - much harder than digits. Use `keras.datasets.fashion_mnist.load_data()`. Aim for the best test accuracy you can.

Hints: more epochs, more units, Dropout.

In [ ]:
# TODO 3: Fashion-MNIST
# (xf_train, yf_train), (xf_test, yf_test) = keras.datasets.fashion_mnist.load_data()
# xf_train = xf_train.reshape(-1, 784).astype('float32') / 255.0
# xf_test  = xf_test.reshape(-1, 784).astype('float32')  / 255.0
#
# model_fm = keras.Sequential([
#     keras.layers.Input(shape=(784,)),
#     keras.layers.Dense(256, activation='relu'),
#     keras.layers.Dropout(0.3),
#     keras.layers.Dense(128, activation='relu'),
#     keras.layers.Dropout(0.3),
#     keras.layers.Dense(10,  activation='softmax'),
# ])
# model_fm.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# history_fm = model_fm.fit(xf_train, yf_train, epochs=15, batch_size=128, validation_split=0.1, verbose=2)
# print('Fashion-MNIST test accuracy:', model_fm.evaluate(xf_test, yf_test, verbose=0)[1])

## End of Day 1

You can now: build a neuron from scratch, explain why hidden layers matter, run gradient descent by hand, and train a Keras MLP that beats 98% on MNIST. Tomorrow: **CNNs** - the same recipe with smarter layers built for images.